# Online Retail — 5a, Câu 12: FP-Growth và so sánh Apriori

**Võ Thiện Linh — MSSV: 6451071041**

Tìm tập phổ biến và luật kết hợp trong hóa đơn Germany bằng FP-Growth (`min_sup = 5%`, `min_conf = 50%`) rồi so sánh với Apriori. Các thuật toán được cài đặt bằng Python, không cần `mlxtend`.

In [ ]:
print('Võ Thiện Linh - MSSV: 6451071041')
from pathlib import Path
from collections import Counter
from itertools import combinations
import math
import pandas as pd
from IPython.display import display
candidates = [Path('Online Retail.xlsx'), Path('../6451071041_LAB1_Bai2_Trang37') / 'Online Retail.xlsx', Path('../Online Retail.xlsx'), Path('LAB1') / 'Phan02' / '6451071041_LAB1_Bai2_Trang37' / 'Online Retail.xlsx', Path('/content/Online Retail.xlsx')]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy Online Retail.xlsx. Thư mục đang chạy: {Path.cwd()}')
df = pd.read_excel(data_path, engine='openpyxl')
df['Description'] = df['Description'].str.strip()
df = df.dropna(subset=['InvoiceNo']).copy()
df['InvoiceNo'] = df['InvoiceNo'].astype('string').str.strip()
df = df.loc[~df['InvoiceNo'].str.contains('C', regex=False, na=False)]
germany = df.loc[df['Country'].eq('Germany')]
grouped = germany.groupby(['InvoiceNo', 'Description'], as_index=False, dropna=True)['Quantity'].sum()
invoice_ids = pd.Index(germany['InvoiceNo'].drop_duplicates(), name='InvoiceNo')
purchases = grouped.loc[grouped['Quantity'].gt(0) & grouped['Description'].ne('POSTAGE')]
items_by_invoice = purchases.groupby('InvoiceNo')['Description'].agg(frozenset).to_dict()
transactions = [items_by_invoice.get(invoice, frozenset()) for invoice in invoice_ids]
def apriori_counts(transactions, min_support):
    minimum_count = math.ceil(min_support * len(transactions))
    counts = Counter(item for transaction in transactions for item in transaction)
    previous = {frozenset([item]): count for item, count in counts.items() if count >= minimum_count}
    all_frequent = dict(previous)
    frequent_items = set(counts)
    size = 2
    while previous:
        previous_list = sorted(previous, key=lambda s: tuple(sorted(s)))
        previous_set = set(previous)
        candidates = set()
        for i, left in enumerate(previous_list):
            for right in previous_list[i + 1:]:
                candidate = left | right
                if len(candidate) == size and all(frozenset(part) in previous_set for part in combinations(candidate, size - 1)):
                    candidates.add(candidate)
        if not candidates:
            break
        candidate_counts = Counter()
        for transaction in transactions:
            for parts in combinations(sorted(transaction & frequent_items), size):
                itemset = frozenset(parts)
                if itemset in candidates:
                    candidate_counts[itemset] += 1
        previous = {itemset: count for itemset, count in candidate_counts.items() if count >= minimum_count}
        all_frequent.update(previous)
        size += 1
    return all_frequent, minimum_count
class FPNode:
    def __init__(self, item, parent=None):
        self.item = item
        self.count = 0
        self.parent = parent
        self.children = {}

def build_fp_tree(weighted_transactions, minimum_count):
    item_counts = Counter()
    for items, weight in weighted_transactions:
        item_counts.update({item: weight for item in set(items)})
    frequent = {item: count for item, count in item_counts.items() if count >= minimum_count}
    if not frequent:
        return None, None, None
    root = FPNode(None)
    headers = {item: [] for item in frequent}
    for items, weight in weighted_transactions:
        ordered = sorted((item for item in set(items) if item in frequent), key=lambda item: (-frequent[item], item))
        current = root
        for item in ordered:
            child = current.children.get(item)
            if child is None:
                child = FPNode(item, current)
                current.children[item] = child
                headers[item].append(child)
            child.count += weight
            current = child
    return root, headers, frequent

def fp_growth(weighted_transactions, minimum_count, prefix=frozenset()):
    root, headers, frequent = build_fp_tree(weighted_transactions, minimum_count)
    if root is None:
        return {}
    result = {}
    for item in sorted(headers, key=lambda value: (frequent[value], value)):
        pattern = prefix | frozenset([item])
        result[pattern] = frequent[item]
        conditional_paths = []
        for node in headers[item]:
            path = []
            parent = node.parent
            while parent is not None and parent.item is not None:
                path.append(parent.item)
                parent = parent.parent
            if path:
                conditional_paths.append((path, node.count))
        if conditional_paths:
            result.update(fp_growth(conditional_paths, minimum_count, pattern))
    return result
def association_rules(itemset_counts, transaction_count, min_confidence):
    rules = []
    for itemset, count in itemset_counts.items():
        if len(itemset) < 2:
            continue
        for size in range(1, len(itemset)):
            for parts in combinations(sorted(itemset), size):
                antecedent = frozenset(parts)
                consequent = itemset - antecedent
                confidence = count / itemset_counts[antecedent]
                if confidence >= min_confidence:
                    support = count / transaction_count
                    lift = confidence / (itemset_counts[consequent] / transaction_count)
                    rules.append({'antecedent': antecedent, 'consequent': consequent, 'support': support, 'confidence': confidence, 'lift': lift})
    return rules
min_support = 0.05
min_confidence = 0.50
apriori_frequent, minimum_count = apriori_counts(transactions, min_support)
fp_frequent = fp_growth([(list(transaction), 1) for transaction in transactions], minimum_count)
apriori_rules = association_rules(apriori_frequent, len(transactions), min_confidence)
fp_rules = association_rules(fp_frequent, len(transactions), min_confidence)
def rule_keys(rules):
    return {(r['antecedent'], r['consequent'], r['support'], r['confidence']) for r in rules}
itemsets_match = apriori_frequent == fp_frequent
rules_match = rule_keys(apriori_rules) == rule_keys(fp_rules)
comparison = pd.DataFrame([{'algorithm':'Apriori', 'frequent_itemsets':len(apriori_frequent), 'association_rules':len(apriori_rules)}, {'algorithm':'FP-Growth', 'frequent_itemsets':len(fp_frequent), 'association_rules':len(fp_rules)}])
print(f'Germany invoices: {len(transactions):,}; min_sup = {min_support:.0%} ({minimum_count:,} invoices); min_conf = {min_confidence:.0%}')
print('Tập phổ biến giống nhau:', itemsets_match)
print('Luật kết hợp giống nhau:', rules_match)
display(comparison)
fp_itemsets = pd.DataFrame([{'itemsets':', '.join(sorted(items)), 'size':len(items), 'support':count / len(transactions), 'invoice_count':count} for items, count in fp_frequent.items()]).sort_values(['size','support'], ascending=[True,False]).reset_index(drop=True)
fp_rules_df = pd.DataFrame([{'antecedent':', '.join(sorted(r['antecedent'])), 'consequent':', '.join(sorted(r['consequent'])), 'support':r['support'], 'confidence':r['confidence'], 'lift':r['lift']} for r in fp_rules])
if not fp_rules_df.empty:
    fp_rules_df = fp_rules_df.sort_values(['confidence','support'], ascending=False).reset_index(drop=True)
print('Tập phổ biến FP-Growth:')
display(fp_itemsets)
print('Luật kết hợp FP-Growth:')
display(fp_rules_df)
assert itemsets_match and rules_match